# Exemplo 1 — Padaria Pão Dourado
## Programação Linear: mix de produção (maximização)

> **Como usar:** abra este notebook no [Google Colab](https://colab.research.google.com) (Arquivo → Fazer upload de notebook, ou arraste este `.ipynb`). Rode as células na ordem com `Shift+Enter`.

## 1. O problema

> A padaria **Pão Dourado** produz duas fornadas diferentes: **pão francês** e **pão doce**.
>
> - Cada fornada de **pão francês** dá um lucro de **R$ 40** e consome **5 kg de farinha** e **2 horas de forno**.
> - Cada fornada de **pão doce** dá um lucro de **R$ 50** e consome **4 kg de farinha** e **3 horas de forno**.
> - Por dia, a padaria dispõe de **100 kg de farinha** e **54 horas de forno** (contando os fornos disponíveis).
>
> **Pergunta:** quantas fornadas de cada tipo devem ser produzidas por dia para **maximizar o lucro**?

## 2. Classificação passo a passo das variáveis

Seguindo a receita de modelagem (ler → variáveis → objetivo → restrições → conferir unidades):

**Passo 1 — Leitura:** dois produtos concorrendo por dois recursos limitados (farinha e forno) — problema clássico de mix de produção.

**Passo 2 — Variáveis de decisão (com unidade!):**
- `x` = fornadas de **pão francês** produzidas por dia
- `y` = fornadas de **pão doce** produzidas por dia

**Passo 3 — Função objetivo** (o que queremos maximizar):

$$\text{Max } Z = 40x + 50y \quad \text{(lucro diário, em R\$)}$$

**Passo 4 — Restrições** (uma por recurso, + não negatividade):

$$5x + 4y \le 100 \quad \text{(farinha, em kg)}$$
$$2x + 3y \le 54 \quad \text{(forno, em horas)}$$
$$x \ge 0, \quad y \ge 0$$

**Passo 5 — Conferência de unidades:** [kg/fornada]·[fornadas] + [kg/fornada]·[fornadas] = kg ✔ (farinha); [h/fornada]·[fornadas] + [h/fornada]·[fornadas] = h ✔ (forno).

## 3. Código — resolvendo com PuLP

In [1]:
# No Google Colab, o PuLP não vem pré-instalado — instale com o comando abaixo (uma vez por sessão)
!pip install pulp -q

In [2]:
from pulp import LpMaximize, LpProblem, LpVariable, LpStatus, value

# --- Passo 2: variáveis de decisão ---
modelo = LpProblem('padaria_pao_dourado', LpMaximize)

x = LpVariable('fornadas_pao_frances', lowBound=0)
y = LpVariable('fornadas_pao_doce', lowBound=0)

# --- Passo 3: função objetivo ---
modelo += 40 * x + 50 * y, 'lucro_diario_R$'

# --- Passo 4: restrições ---
modelo += 5 * x + 4 * y <= 100, 'farinha_kg'
modelo += 2 * x + 3 * y <= 54, 'forno_horas'

# --- resolver ---
modelo.solve()

print('Status              :', LpStatus[modelo.status])
print('Fornadas pão francês:', x.value())
print('Fornadas pão doce   :', y.value())
print('Lucro máximo        : R$', value(modelo.objective))
print()
for nome, restricao in modelo.constraints.items():
    folga = -restricao.slack  # slack negativo do PuLP -> sobra positiva do recurso
    print(f'Restrição {nome}: sobra de recurso = {folga:.2f}')

Status              : Optimal
Fornadas pão francês: 12.0
Fornadas pão doce   : 10.0
Lucro máximo        : R$ 980.0

Restrição farinha_kg: sobra de recurso = 0.00
Restrição forno_horas: sobra de recurso = 0.00


## 4. Leia o resultado

**Esperado:** x = 12 fornadas de pão francês, y = 10 fornadas de pão doce, **lucro máximo = R$ 980/dia**, com farinha e forno totalmente consumidos (sobra ≈ 0 nas duas restrições — ambos os recursos ficam **ativos**).

**Para pensar:** e se a padaria conseguisse mais 10 kg de farinha por dia (chegando a 110 kg)? O lucro aumentaria — mas quanto, exatamente, por kg extra? Essa pergunta é o coração da **Análise de Sensibilidade**.



----
# Atividade Fotos

VM Small: 80 imagens/min, 8GB memoria, U$1,00/hora  

VM Large: 150 imagens/min, 16GB memoria, U$1,50/hora

Disponivel: 96GB memoria, orcamento U$10,00/hora


In [4]:
from pulp import LpMaximize, LpProblem, LpVariable, LpStatus, value

# --- Passo 2: variáveis de decisão ---
modelo = LpProblem('processamento_imgs', LpMaximize)

x = LpVariable('vm_small', lowBound=0)
y = LpVariable('vm_large', lowBound=0)

# --- Passo 3: função objetivo ---
modelo += 80 * x + 150 * y, 'imagens diarias'

# --- Passo 4: restrições ---
modelo += 8 * x + 16 * y <= 96, 'armazenamento'
modelo += 1 * x + 1.5 * y <= 10, 'dolar_hora'

# --- resolver ---
modelo.solve()

print('Status              :', LpStatus[modelo.status])
print('VMs Small.          :', x.value())
print('VMs Large           :', y.value())
print('Fotos maximo        :', value(modelo.objective))
print()
for nome, restricao in modelo.constraints.items():
    folga = -restricao.slack  # slack negativo do PuLP -> sobra positiva do recurso
    print(f'Restrição {nome}: sobra de recurso = {folga:.2f}')

Status              : Optimal
VMs Small.          : 4.0
VMs Large           : 4.0
Fotos maximo        : 920.0

Restrição armazenamento: sobra de recurso = 0.00
Restrição dolar_hora: sobra de recurso = 0.00


---
# Suplementos

Dose A: R$3,00, 4g de proteina, 2g de carboidrato

Dose B: R$5,00, 2g de proteina, 6g de carboidrato

Exigencia: minimo 20g de proteina e 30g carboidrato

In [7]:
from pulp import LpMinimize, LpProblem, LpVariable, LpStatus, value

# --- Passo 2: variáveis de decisão ---
modelo = LpProblem('processamento_imgs', LpMinimize)

x = LpVariable('dose_a', lowBound=0)
y = LpVariable('dose_b', lowBound=0)

# --- Passo 3: função objetivo ---
modelo += 3.0 * x + 5.0 * y, 'custo_dose'

# --- Passo 4: restrições ---
modelo += 4 * x + 2 * y >= 20, 'proteina'
modelo += 2 * x + 6 * y >= 30, 'carboidrato'

# --- resolver ---
modelo.solve()

print('Status              :', LpStatus[modelo.status])
print('Doses A.            :', x.value())
print('Doses B             :', y.value())
print('Preco minimo        :', value(modelo.objective))
print()
for nome, restricao in modelo.constraints.items():
    folga = -restricao.slack  # slack negativo do PuLP -> sobra positiva do recurso
    print(f'Restrição {nome}: sobra de recurso = {folga:.2f}')

Status              : Optimal
Doses A.            : 3.0
Doses B             : 4.0
Preco minimo        : 29.0

Restrição proteina: sobra de recurso = 0.00
Restrição carboidrato: sobra de recurso = 0.00
